In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import numpy as np
import sklearn

In [24]:
def extract_require_statements(parquet_file_path):
    df = pd.read_parquet(parquet_file_path)
    df_filtered = df[df['failure_invariant'].notna()].copy()
    df_filtered['failure_invariant_str'] = df_filtered['failure_invariant'].astype(str).str.lower().str.strip()
    revert_rows = df_filtered[df_filtered['failure_invariant_str'].str.contains('require', na=False)]
    return revert_rows['failure_invariant_str'].tolist()


In [23]:
import re

def extract_conditions(parquet_file_path):
    cleaned_require_statements = []
    require_statements = extract_require_statements(parquet_file_path)
    
    for stmt in require_statements:
        stmt = stmt.strip()

        # Extract content inside require(...)
        match = re.match(r'^require\((.*)\);?$', stmt)
        if match:
            inner = match.group(1).strip()
            condition = extract_before_top_level_comma(inner)
            cleaned_require_statements.append(condition)

    return cleaned_require_statements

def extract_before_top_level_comma(s):
    """Extracts everything before the first comma at the top level (not inside nested parentheses)."""
    depth = 0
    for i, char in enumerate(s):
        if char == '(':
            depth += 1
        elif char == ')':
            depth -= 1
        elif char == ',' and depth == 0:
            return s[:i].strip()
    return s.strip()  # No comma found outside nested parentheses


In [41]:
import ast
from typing import List

def parse_snippets_to_asts(snippets: List[str]) -> List[ast.AST]:
    """
    Parses a list of Solidity code snippets directly into Python ASTs.
    No transformations are done — this assumes the syntax is parseable by Python.
    """
    asts = []
    for code in snippets:
        try:
            node = ast.parse(code)
            asts.append(node)
        except SyntaxError as e:
            print(f"Skipping snippet due to parse error: {code}")
            continue
    return asts


# DBSCAN

- Jaccard Distance, Cosine Distance, AST edit distance with TF-IDF vectors


## Jaccard Distance

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import pairwise_distances


def dbscan_cluster_jaccard(invariants, eps=0.5, min_samples=5, metric = 'cosine'):
    """
    DBSCAN with jaccard 
    """
    vectorizer = CountVectorizer(binary=True)
    X = vectorizer.fit_transform(invariants).toarray()  # dense array
    distance_matrix = pairwise_distances(X, metric='jaccard')

    db = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    db.fit(distance_matrix)
    cluster_dict = {}
    for label, invariant in zip(db.labels_, invariants):
        if label == -1:
            continue
        if label not in cluster_dict:
            cluster_dict[label] = []
        cluster_dict[label].append(invariant)
    
    return cluster_dict


## Cosine Distance

In [9]:
def dbscan_cluster_cosine(invariants, eps=0.5, min_samples=5, metric = 'cosine'):
    """
    DBSCAN with cosine 
    """
    vectorizer = TfidfVectorizer()
    invariant_vectors = vectorizer.fit_transform(invariants)
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    db.fit(invariant_vectors)
    cluster_dict = {}
    for label, invariant in zip(db.labels_, invariants):
        if label == -1:
            continue
        if label not in cluster_dict:
            cluster_dict[label] = []
        cluster_dict[label].append(invariant)
    
    return cluster_dict


## AST with TF-IDF vectors
- harder than thought to change it into AST syntax

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

def dbscan_with_ast_tfidf(invariants, eps=0.5, min_samples=2):
    ast_texts = parse_snippets_to_asts(invariants)
   
    vectorizer = TfidfVectorizer()
    X_tfidf = vectorizer.fit_transform(ast_texts)
    
    distance_matrix = pairwise_distances(X_tfidf, metric='cosine')
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    clustering.fit(distance_matrix)
    cluster_dict = {}
    for label, invariant in zip(clustering.labels_, invariants):
        if label == -1:
            continue
        if label not in cluster_dict:
            cluster_dict[label] = []
        cluster_dict[label].append(invariant)

    return cluster_dict


## Run it

In [46]:
file_path = "../datasets/analysis/MS_All_20000.parquet"
#conditions = extract_conditions(parquet_file_path)
conditions = extract_require_statements(file_path)
conditions = list(set(conditions))
print(len(conditions))
labels = dbscan_with_ast_tfidf(conditions, eps=0.01, min_samples=5)
"""
for cluster_label, statements in labels.items():
    print(f"Cluster {cluster_label}:")
    counter = 0
    for statement in statements:
        print(f"  {statement}")
    print("\n")
"""

547
Skipping snippet due to parse error: require(from == owner(), "trading is not enabled");
Skipping snippet due to parse error: require(!paused() || hasexception(_msgsender()), "pausable: paused (and no exception)");
Skipping snippet due to parse error: require( value <= allowed[from][msg.sender], "erc20: transfer amount exceeds allowance" );
Skipping snippet due to parse error: require(univ2pairaddress != address(0) || excluded, "liquidity pair not yet created.");
Skipping snippet due to parse error: require( launch != 0 && amount <= maxtxamount, "launch / max txamount 2% at launch" );
Skipping snippet due to parse error: require(!address.iscontract(_to) || !address.iscontract(_from), 'ryoshi says: no bots allowed!');
Skipping snippet due to parse error: require(v2pair != address(0) || excluded, "liquidity pair not yet created.");
Skipping snippet due to parse error: require(!isbot[from], "error");
Skipping snippet due to parse error: require( value <= _balanceof(from), "erc20: tran

AttributeError: 'Module' object has no attribute 'lower'

In [ ]:
# find best dinstance function --> semantic distance!!!
# then cluster based on that
# clustering programs (ie: clone detection) 
# use server for clustering

- BERT encoding distance
- jaccard distance, levensthein
- other SEMANTIC distances (NL)

# COCLUBERT

## PDG Analysis